# ACE-Step 1.5 - Google Colab A100

Bu notebook A100 GPU icin hazirlandi. En kaliteli yerel ayar olarak `acestep-v15-xl-sft` DiT modeli ve `acestep-5Hz-lm-4B` LM modeli kullanir.

A100 ile hedef kapasite:
- Maksimum sure: proje limitine gore 600 saniye, yani 10 dakika.
- Rahat baslangic: 120-240 saniye.
- Kalite modu: XL SFT + 4B LM + vLLM backend.
- Hizi arttirmak icin: `acestep-v15-xl-turbo` veya `acestep-v15-turbo` secilebilir.

Colab menusu: `Runtime > Change runtime type > GPU > A100` secili olmali.

In [ ]:
#@title 1) GPU kontrolu
!nvidia-smi

import subprocess

gpu_name = subprocess.check_output(
    "nvidia-smi --query-gpu=name --format=csv,noheader", shell=True
).decode().strip()
print("\nAktif GPU:", gpu_name)

if "A100" not in gpu_name:
    print("UYARI: Bu notebook A100 icin optimize edildi. Yine calisir ama ayarlari dusurmen gerekebilir.")

In [ ]:
#@title 2) Repo ve bagimlilik kurulumu
# Ilk calistirmada uzun surebilir. Modeller de ilk uretimde indirilecektir.

%cd /content
!curl -LsSf https://astral.sh/uv/install.sh | sh
!test -d ACE-Step-1.5 || git clone https://github.com/ACE-Step/ACE-Step-1.5.git
%cd /content/ACE-Step-1.5
!/root/.local/bin/uv sync

print("Kurulum tamam.")

In [ ]:
#@title 3) A100 kalite ayarlari ve modelleri hazirla
import os
import site
import sys
from pathlib import Path

PROJECT_ROOT = Path("/content/ACE-Step-1.5")
CHECKPOINTS_DIR = Path("/content/acestep_checkpoints")
OUTPUT_DIR = Path("/content/acestep_outputs")
VENV_DIR = PROJECT_ROOT / ".venv"

# Colab notebook kernel'i, `uv sync` ile kurulan .venv paketlerini otomatik gormez.
# Bu kopru loguru/torch/gradio gibi tum proje bagimliliklarini aktif kernel'e tanitir.
site_packages = next(VENV_DIR.glob("lib/python*/site-packages"), None)
if site_packages is None:
    raise RuntimeError(".venv site-packages bulunamadi. Once 2. kurulum hucresini calistir.")
site.addsitedir(str(site_packages))
if str(site_packages) in sys.path:
    sys.path.remove(str(site_packages))
sys.path.insert(0, str(site_packages))
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["ACESTEP_PROJECT_ROOT"] = str(PROJECT_ROOT)
os.environ["ACESTEP_CHECKPOINTS_DIR"] = str(CHECKPOINTS_DIR)
os.environ["ACESTEP_INIT_LLM"] = "true"
os.environ["ACESTEP_LM_BACKEND"] = "vllm"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project:", PROJECT_ROOT)
print("Models:", CHECKPOINTS_DIR)
print("Outputs:", OUTPUT_DIR)

In [ ]:
#@title 4) Servisi baslat: A100 en iyi kalite
# Ilk defa calistirirken XL DiT ve 4B LM indirir. Bu dosyalar buyuktur.

from acestep.handler import AceStepHandler
from acestep.llm_inference import LLMHandler
from acestep.model_downloader import ensure_dit_model, ensure_lm_model, get_checkpoints_dir

DIT_MODEL = "acestep-v15-xl-sft"       # En iyi kalite. Daha hizli: acestep-v15-xl-turbo
LM_MODEL = "acestep-5Hz-lm-4B"         # A100 icin en iyi LM secimi
BACKEND = "vllm"
DEVICE = "cuda"

checkpoints = get_checkpoints_dir(str(CHECKPOINTS_DIR))

ok, msg = ensure_dit_model(DIT_MODEL, checkpoints)
print(msg)
if not ok:
    raise RuntimeError(msg)

ok, msg = ensure_lm_model(LM_MODEL, checkpoints)
print(msg)
if not ok:
    raise RuntimeError(msg)

dit_handler = AceStepHandler()
init_status, can_generate = dit_handler.initialize_service(
    project_root=str(PROJECT_ROOT),
    checkpoint_dir=str(CHECKPOINTS_DIR),
    config_path=DIT_MODEL,
    device=DEVICE,
    dtype="bfloat16",
    torch_compile=True,
    offload_to_cpu=False,
    offload_dit_to_cpu=False,
)
print(init_status)
if not can_generate:
    raise RuntimeError("DiT modeli baslatilamadi.")

llm_handler = LLMHandler()
llm_handler.initialize(
    checkpoint_dir=str(CHECKPOINTS_DIR),
    lm_model_path=LM_MODEL,
    backend=BACKEND,
    device=DEVICE,
)

print("ACE-Step A100 servisi hazir.")

In [ ]:
#@title 5) Prompt, sarki sozu ve sure gir
prompt = "Turkish pop ballad, emotional male vocal, modern cinematic arrangement, piano, strings, warm bass, radio-ready mix" #@param {type:"string"}
lyrics = """[Verse 1]\nGece iner usul usul sokaklara\nKalbim yine seni sorar aynalara\n\n[Chorus]\nGel desem duyulur mu sesim\nYollar sana cikar nefesim\n""" #@param {type:"string"}
duration_seconds = 180 #@param {type:"slider", min:10, max:600, step:10}
vocal_language = "tr" #@param ["tr", "en", "de", "fr", "es", "it", "ja", "ko", "zh", "unknown"]
bpm = 0 #@param {type:"integer"}
keyscale = "" #@param {type:"string"}
batch_size = 1 #@param {type:"slider", min:1, max:8, step:1}
seed = -1 #@param {type:"integer"}
use_lm_thinking = True #@param {type:"boolean"}
format_prompt_and_lyrics = True #@param {type:"boolean"}
audio_format = "flac" #@param ["flac", "wav", "mp3"]

duration_seconds = max(10, min(600, int(duration_seconds)))
bpm_value = None if int(bpm) <= 0 else int(bpm)
lyrics_value = lyrics.strip() if lyrics.strip() else "[Instrumental]"

print("Prompt:", prompt)
print("Sure:", duration_seconds, "saniye")
print("Cikti sayisi:", batch_size)

In [ ]:
#@title 6) Muzik uret
from IPython.display import Audio, display
from acestep.inference import GenerationParams, GenerationConfig, format_sample, generate_music

caption_value = prompt
keyscale_value = keyscale.strip()
use_cot_metas = True

if format_prompt_and_lyrics and llm_handler is not None:
    formatted = format_sample(
        llm_handler=llm_handler,
        caption=prompt,
        lyrics=lyrics_value,
        user_metadata={"duration": duration_seconds, "language": vocal_language},
    )
    if formatted.success:
        caption_value = formatted.caption or caption_value
        lyrics_value = formatted.lyrics or lyrics_value
        bpm_value = bpm_value or formatted.bpm
        keyscale_value = keyscale_value or (formatted.keyscale or "")
        if formatted.duration:
            duration_seconds = int(min(600, max(10, float(formatted.duration))))
        use_cot_metas = False
        print("Prompt/soz LM ile formatlandi.")
    else:
        print("Formatlama atlandi:", formatted.error)

params = GenerationParams(
    task_type="text2music",
    caption=caption_value,
    lyrics=lyrics_value,
    duration=duration_seconds,
    bpm=bpm_value,
    keyscale=keyscale_value,
    vocal_language=vocal_language,
    inference_steps=32,
    seed=seed,
    thinking=use_lm_thinking,
    use_cot_metas=use_cot_metas,
    use_cot_caption=use_lm_thinking,
    use_cot_language=use_lm_thinking,
    use_constrained_decoding=True,
)

config = GenerationConfig(
    batch_size=batch_size,
    audio_format=audio_format,
    use_random_seed=(seed < 0),
    allow_lm_batch=True,
    lm_batch_chunk_size=8,
)

result = generate_music(dit_handler, llm_handler, params, config, save_dir=str(OUTPUT_DIR))

if not result.success:
    raise RuntimeError(result.error)

print("Uretim tamam. Dosyalar:")
for audio in result.audios:
    path = audio.get("path") or audio.get("audio_path")
    print(path)
    if path:
        display(Audio(path))

In [ ]:
#@title Opsiyonel: Gradio arayuzunu public link ile ac
# Python API yerine web arayuzu kullanmak istersen bu hucreyi calistir.
# Durdurmak icin Colab runtime'i interrupt edebilirsin.

%cd /content/ACE-Step-1.5
!/root/.local/bin/uv run acestep --share --server-name 0.0.0.0 --init_service true --config_path acestep-v15-xl-sft --lm_model_path acestep-5Hz-lm-4B --init_llm true